In [1]:
import pandas as pd
import os 
import numpy as np

import gurobipy as gp
from gurobipy import GRB

dir = "/Users/rahulprakash/Desktop/Exam_Scheduling_Sp24/2023-2024"
os.chdir(dir)
os.listdir()

['Final_Exams2023.csv',
 '.DS_Store',
 '2023.ipynb',
 'StudentRegistration2023.csv',
 'Schedule2023.csv',
 'Class_Info2023.csv']

In [2]:
class_info2023 = pd.read_csv('Class_Info2023.csv')
registrar_schedule2023 = pd.read_csv("Final_Exams2023.csv") #The schedule the registrar implemented (used for comparison purposes)
student_registration2023 = pd.read_csv("StudentRegistration2023.csv") #registration for each student class schedule
exam_info2023 = pd.read_csv("Schedule2023.csv") #information on each class room enrollment


In [3]:
class_info2023['TIME_Column'] = pd.to_datetime(class_info2023['TIME'], format='%H:%M').dt.time
class_info2023

,BUILDING_CODE,BUILDING,ROOM_NUMBER,DAY_OF_WEEK,TIME,AVAILABLE_TO_SCHEDULE,ROOM_CAPACITY,ROOM_TYPE,TIME_Column
0,RZR,Rayzor Hall,227,U,06:50,Y,12,Classroom - Flexible/SCALAR,06:50:00
1,RZR,Rayzor Hall,227,U,07:00,Y,12,Classroom - Flexible/SCALAR,07:00:00
2,RZR,Rayzor Hall,227,U,07:10,Y,12,Classroom - Flexible/SCALAR,07:10:00
3,RZR,Rayzor Hall,227,U,07:20,Y,12,Classroom - Flexible/SCALAR,07:20:00
4,RZR,Rayzor Hall,227,U,07:30,Y,12,Classroom - Flexible/SCALAR,07:30:00
...,...,...,...,...,...,...,...,...,...
103098,PCF,Provisional Campus Facility,4,W,11:50,Y,90,Classroom - Flexible/SCALAR,11:50:00
103099,PCF,Provisional Campus Facility,4,W,12:00,Y,90,Classroom - Flexible/SCALAR,12:00:00
103100,PCF,Provisional Campus Facility,4,W,12:10,Y,90,Classroom - Flexible/SCALAR,12:10:00
103101,PCF,Provisional Campus Facility,4,W,12:20,Y,90,Classroom - Flexible/SCALAR,12:20:00


In [4]:
class_info2023['TIME_FLOAT'] = class_info2023['TIME_Column'].apply(lambda x: float(f"{x.hour}.{x.minute:02d}"))
filtered_df = class_info2023[(class_info2023['TIME_FLOAT'] == 9.0) | (class_info2023['TIME_FLOAT'] == 14.0) | (class_info2023['TIME_FLOAT'] == 19.0)]
available_slots = filtered_df[(filtered_df['AVAILABLE_TO_SCHEDULE'] == 'Y')]


available_slots

,BUILDING_CODE,BUILDING,ROOM_NUMBER,DAY_OF_WEEK,TIME,AVAILABLE_TO_SCHEDULE,ROOM_CAPACITY,ROOM_TYPE,TIME_Column,TIME_FLOAT
13,RZR,Rayzor Hall,227,U,09:00,Y,12,Classroom - Flexible/SCALAR,09:00:00,9.0
43,RZR,Rayzor Hall,227,U,14:00,Y,12,Classroom - Flexible/SCALAR,14:00:00,14.0
73,RZR,Rayzor Hall,227,U,19:00,Y,12,Classroom - Flexible/SCALAR,19:00:00,19.0
156,RZR,Rayzor Hall,227,W,09:00,Y,12,Classroom - Flexible/SCALAR,09:00:00,9.0
186,RZR,Rayzor Hall,227,W,14:00,Y,12,Classroom - Flexible/SCALAR,14:00:00,14.0
...,...,...,...,...,...,...,...,...,...,...
102509,PCF,Provisional Campus Facility,4,R,09:00,Y,90,Classroom - Flexible/SCALAR,09:00:00,9.0
102539,PCF,Provisional Campus Facility,4,R,14:00,Y,90,Classroom - Flexible/SCALAR,14:00:00,14.0
102795,PCF,Provisional Campus Facility,4,T,09:00,Y,90,Classroom - Flexible/SCALAR,09:00:00,9.0
102825,PCF,Provisional Campus Facility,4,T,14:00,Y,90,Classroom - Flexible/SCALAR,14:00:00,14.0


In [5]:
mapping = {'M': 1,
           'T': 2,
           'W': 3,
           'R': 4,
           'F': 5,
           'S': 6,
           'U': 7
}

mapping1 = {9.0: 'Morning',
            14.0: 'Afternoon',
            19.0: 'Evening'}

available_slots['DAY_OF_WEEK'] = available_slots['DAY_OF_WEEK'].map(mapping)
available_slots['TIME_OF_DAY'] = available_slots['TIME_FLOAT'].map(mapping1)
available_slots

/var/folders/xy/4pwp0wc90bd548jjl3c1y5mc0000gn/T/ipykernel_79229/4004836996.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  available_slots['DAY_OF_WEEK'] = available_slots['DAY_OF_WEEK'].map(mapping)
/var/folders/xy/4pwp0wc90bd548jjl3c1y5mc0000gn/T/ipykernel_79229/4004836996.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  available_slots['TIME_OF_DAY'] = available_slots['TIME_FLOAT'].map(mapping1)


,BUILDING_CODE,BUILDING,ROOM_NUMBER,DAY_OF_WEEK,TIME,AVAILABLE_TO_SCHEDULE,ROOM_CAPACITY,ROOM_TYPE,TIME_Column,TIME_FLOAT,TIME_OF_DAY
13,RZR,Rayzor Hall,227,7,09:00,Y,12,Classroom - Flexible/SCALAR,09:00:00,9.0,Morning
43,RZR,Rayzor Hall,227,7,14:00,Y,12,Classroom - Flexible/SCALAR,14:00:00,14.0,Afternoon
73,RZR,Rayzor Hall,227,7,19:00,Y,12,Classroom - Flexible/SCALAR,19:00:00,19.0,Evening
156,RZR,Rayzor Hall,227,3,09:00,Y,12,Classroom - Flexible/SCALAR,09:00:00,9.0,Morning
186,RZR,Rayzor Hall,227,3,14:00,Y,12,Classroom - Flexible/SCALAR,14:00:00,14.0,Afternoon
...,...,...,...,...,...,...,...,...,...,...,...
102509,PCF,Provisional Campus Facility,4,4,09:00,Y,90,Classroom - Flexible/SCALAR,09:00:00,9.0,Morning
102539,PCF,Provisional Campus Facility,4,4,14:00,Y,90,Classroom - Flexible/SCALAR,14:00:00,14.0,Afternoon
102795,PCF,Provisional Campus Facility,4,2,09:00,Y,90,Classroom - Flexible/SCALAR,09:00:00,9.0,Morning
102825,PCF,Provisional Campus Facility,4,2,14:00,Y,90,Classroom - Flexible/SCALAR,14:00:00,14.0,Afternoon


In [11]:
#read all of the data and start the new version

In [6]:
person_schedule_map = {}

# Iterating through each row in the DataFrame
for index, row in student_registration2023.iterrows():
    person_id = row['PERSON_IDENTIFIER']
    course_code = row['SUBJECT'] + str(row['COURSE_NUMBER'])
    
    # Check if the person_id is already in the dictionary
    if person_id in person_schedule_map:
        # If present, append the course_code to the existing tuple
        person_schedule_map[person_id].add(course_code)
    else:
        # If not present, create a new entry in the dictionary
        person_schedule_map[person_id] = {course_code}

# To convert the sets to tuples for the final output as specified
person_schedule_map = {k: tuple(v) for k, v in person_schedule_map.items()}

person_schedule_map

{93423: ('THEA303',),
 94572: ('MUSI335',),
 361383: ('PORT106',),
 559884: ('JAPA141',),
 565527: ('FWIS184', 'RELI421'),
 566199: ('SOCI325', 'ENST312', 'PHIL400', 'LPAP171', 'LPAP183', 'HIST340'),
 607599: ('HIST438', 'HART358'),
 637395: ('ANTH200',),
 883479: ('JAPA141',),
 971649: ('PSYC202', 'KINE498', 'KINE302', 'KINE321'),
 975462: ('ASIA223',),
 1074561: ('HART203', 'COMP140', 'ENGL210', 'MATH101', 'FWIS100'),
 1107495: ('KINE498', 'KINE375', 'NAVA302', 'SMGT450', 'KINE302', 'KINE321'),
 1167519: ('MATH321', 'MATH356'),
 1235406: ('STAT413', 'STAT482', 'STAT487'),
 1292541: ('FREN263',),
 1305336: ('MUSI426', 'MUSI414'),
 1365753: ('KORE401',),
 1368495: ('COMP382', 'COMP480', 'COMP490', 'COMP310'),
 1370427: ('ECON483', 'ECON300', 'BUSI380', 'BUSI440', 'ECON441'),
 1378170: ('ARCH450',),
 1389207: ('GERM301',),
 1389840: ('PHYS311', 'CMOR420', 'HIST237', 'PHYS301'),
 1396599: ('ARCH450', 'ARCH327'),
 1423698: ('SMGT364', 'ASTR101', 'SOSC302', 'SMGT360', 'SMGT450', 'SOCI102')

In [7]:
schedule_counts = {}

for schedule in person_schedule_map.values():
    # Sort the schedules to ensure that the same courses in different orders are considered the same
    sorted_schedule = tuple(sorted(schedule))
    if sorted_schedule in schedule_counts:
        schedule_counts[sorted_schedule] += 1
    else:
        schedule_counts[sorted_schedule] = 1

# Count of unique schedules
num_of_unique_schedules = len(schedule_counts)
print(schedule_counts)
print(num_of_unique_schedules)

{('THEA303',): 1, ('MUSI335',): 2, ('PORT106',): 2, ('JAPA141',): 5, ('FWIS184', 'RELI421'): 1, ('ENST312', 'HIST340', 'LPAP171', 'LPAP183', 'PHIL400', 'SOCI325'): 1, ('HART358', 'HIST438'): 1, ('ANTH200',): 1, ('KINE302', 'KINE321', 'KINE498', 'PSYC202'): 1, ('ASIA223',): 1, ('COMP140', 'ENGL210', 'FWIS100', 'HART203', 'MATH101'): 1, ('KINE302', 'KINE321', 'KINE375', 'KINE498', 'NAVA302', 'SMGT450'): 1, ('MATH321', 'MATH356'): 1, ('STAT413', 'STAT482', 'STAT487'): 1, ('FREN263',): 5, ('MUSI414', 'MUSI426'): 2, ('KORE401',): 1, ('COMP310', 'COMP382', 'COMP480', 'COMP490'): 1, ('BUSI380', 'BUSI440', 'ECON300', 'ECON441', 'ECON483'): 1, ('ARCH450',): 18, ('GERM301',): 1, ('CMOR420', 'HIST237', 'PHYS301', 'PHYS311'): 1, ('ARCH327', 'ARCH450'): 1, ('ASTR101', 'SMGT360', 'SMGT364', 'SMGT450', 'SOCI102', 'SOSC302'): 1, ('ENGL382',): 1, ('MUSI334',): 1, ('HUMA315', 'LING327', 'PSYC308', 'PSYC340'): 1, ('ARCH311', 'ARCH327'): 1, ('ARCH327',): 4, ('RELI334',): 1, ('BUSI401', 'ECON210', 'ECON481

In [8]:
exam_df = exam_info2023[exam_info2023['EXAM_TYPE'].isin(['Scheduled Final Exam-OTR Room', 'Scheduled Final Exam-Dept Room', 'Scheduled Online Final Exam'])]
exam_demand = {}
for index, row in exam_df.iterrows():
    exam_key = row['SUBJECT'] + str(row['COURSE_NUMBER'])
    if exam_key in exam_demand.keys():
        exam_demand[exam_key] += row['SECTION_ENROLLMENT']
    else:
       exam_demand[exam_key] = row['SECTION_ENROLLMENT'] 

online_exams = []
in_person_exams = []
for index, row in exam_df.iterrows():
    exam_key = row['SUBJECT'] + str(row['COURSE_NUMBER'])
    if row['EXAM_TYPE'] == 'Scheduled Online Final Exam':
        online_exams.append(exam_key)
    else:
        in_person_exams.append(exam_key)

set_of_exams = []
for item in online_exams:
    set_of_exams.append(item)
for item in in_person_exams:
    set_of_exams.append(item)
    
print(in_person_exams)
print(online_exams)

['ANTH205', 'ASIA322', 'ASTR470', 'ASTR451', 'ASTR101', 'ASTR350', 'ASTR102', 'BIOE252', 'BIOE451', 'BIOE440', 'BIOE422', 'BIOE420', 'BIOE341', 'BIOE446', 'BIOE383', 'BIOS201', 'BIOS301', 'BIOS312', 'BIOS341', 'BIOS352', 'BIOS300', 'BIOS372', 'BIOS312', 'BIOS385', 'BIOS329', 'BIOS122', 'BIOS353', 'BIOS441', 'BIOS344', 'CHEM430', 'CHEM211', 'CHEM201', 'CHEM401', 'CHEM121', 'CHEM121', 'CHEM121', 'CHEM452', 'CHEM462', 'CHEM496', 'CHEM219', 'CHEM313', 'COMP450', 'COMP326', 'COMP311', 'COMP490', 'COMP390', 'COMP301', 'COMP417', 'COMP310', 'COMP310', 'COMP442', 'COMP480', 'COMP182', 'COMP413', 'COMP431', 'COMP429', 'COMP462', 'COMP416', 'ECON307', 'ECON437', 'ECON203', 'ECON308', 'ECON210', 'ECON300', 'ECON305', 'ECON200', 'ECON483', 'ECON200', 'ECON203', 'ECON497', 'ECON209', 'ECON209', 'ECON209', 'ECON209', 'ECON209', 'ECON481', 'ECON307', 'ELEC261', 'ELEC326', 'ELEC450', 'ELEC303', 'ELEC434', 'ELEC429', 'ELEC361', 'ENGI302', 'ENST437', 'ENST315', 'FREN424', 'HEAL103', 'HEAL103', 'HIST391'

In [9]:
room_capacity = {}
for index, row in class_info2023.iterrows():
    room = row['BUILDING_CODE'] + str(row['ROOM_NUMBER'])
    room_capacity[room] = row['ROOM_CAPACITY']

room_capacity['Online'] = float('inf')

print(room_capacity)

{'RZR227': 12, 'RZR302': 12, 'RZR304': 20, 'RZR205': 20, 'RZR305': 20, 'RZR310': 12, 'SEW133': 24, 'SEW301': 168, 'SEW101': 36, 'SEW305': 45, 'SEW307': 45, 'SEW303': 45, 'SEW460': 20, 'SEW462': 17, 'SEW309': 82, 'ABL131': 140, 'ANH117': 59, 'ABL130': 52, 'BKH116': 37, 'BKH229': 24, 'BKH102': 37, 'BRK103': 30, 'DCH1042': 46, 'BKH233': 24, 'BRC282': 66, 'BRC285': 24, 'BRC286': 32, 'DBH180': 87, 'DCH1046': 46, 'BRC284': 50, 'BRK101': 150, 'DCC113': 99, 'GRBW212': 32, 'HBH227': 46, 'DCH1064': 73, 'DCH1020 / SYM II': 66, 'GRW220': 26, 'HBH423': 16, 'DCH1070': 73, 'GRBW211': 32, 'GRW160A': 30, 'DCH1055': 240, 'DCH1075': 35, 'HRG224': 22, 'HRZ211': 20, 'HRZ212': 123, 'HBH453': 16, 'HRG125': 24, 'HBHB21': 24, 'HRG126': 24, 'HRZ210': 125, 'HRZAMP': 300, 'HBH427': 27, 'HRG100': 197, 'HRG128': 7, 'HUM327': 24, 'KCK101': 30, 'HUM117': 75, 'HUM118': 30, 'HUM120': 17, 'HUM226': 24, 'HUM328': 42, 'KCK102': 69, 'HUM119': 42, 'HUM227': 24, 'KCK100': 265, 'KWG100': 68, 'LVCCOMMNS': 184, 'KCK107': 20, 'K

In [10]:
in_person_slots = []
for index, row in available_slots.iterrows():
    slot = f"{row['BUILDING_CODE']}{row['ROOM_NUMBER']}_Day{row['DAY_OF_WEEK']}"
    in_person_slots.append((slot, row['TIME_OF_DAY']))
in_person_slots

[('RZR227_Day7', 'Morning'),
 ('RZR227_Day7', 'Afternoon'),
 ('RZR227_Day7', 'Evening'),
 ('RZR227_Day3', 'Morning'),
 ('RZR227_Day3', 'Afternoon'),
 ('RZR227_Day3', 'Evening'),
 ('RZR302_Day5', 'Morning'),
 ('RZR302_Day5', 'Evening'),
 ('RZR302_Day1', 'Morning'),
 ('RZR302_Day1', 'Evening'),
 ('RZR302_Day4', 'Morning'),
 ('RZR302_Day4', 'Evening'),
 ('RZR302_Day6', 'Morning'),
 ('RZR302_Day6', 'Afternoon'),
 ('RZR302_Day6', 'Evening'),
 ('RZR302_Day2', 'Morning'),
 ('RZR302_Day2', 'Evening'),
 ('RZR302_Day7', 'Morning'),
 ('RZR302_Day7', 'Afternoon'),
 ('RZR302_Day7', 'Evening'),
 ('RZR302_Day3', 'Morning'),
 ('RZR302_Day3', 'Evening'),
 ('RZR304_Day5', 'Morning'),
 ('RZR304_Day5', 'Afternoon'),
 ('RZR304_Day5', 'Evening'),
 ('RZR304_Day1', 'Morning'),
 ('RZR205_Day3', 'Morning'),
 ('RZR205_Day3', 'Afternoon'),
 ('RZR205_Day3', 'Evening'),
 ('RZR227_Day5', 'Morning'),
 ('RZR227_Day5', 'Afternoon'),
 ('RZR227_Day5', 'Evening'),
 ('RZR227_Day1', 'Morning'),
 ('RZR227_Day1', 'Afternoon')

In [11]:
online_slots = []

for day in range(1, 8):
    online_day = f"Online_Day{day}"
    online_slots.append((online_day, 'Morning'))
    online_slots.append((online_day, 'Afternoon'))
    online_slots.append((online_day, 'Evening'))

all_slots = []
for item in in_person_slots:
    all_slots.append(item)
for item in online_slots:
    all_slots.append(item)

print(in_person_slots)
print(online_slots)
print(all_slots)

[('RZR227_Day7', 'Morning'), ('RZR227_Day7', 'Afternoon'), ('RZR227_Day7', 'Evening'), ('RZR227_Day3', 'Morning'), ('RZR227_Day3', 'Afternoon'), ('RZR227_Day3', 'Evening'), ('RZR302_Day5', 'Morning'), ('RZR302_Day5', 'Evening'), ('RZR302_Day1', 'Morning'), ('RZR302_Day1', 'Evening'), ('RZR302_Day4', 'Morning'), ('RZR302_Day4', 'Evening'), ('RZR302_Day6', 'Morning'), ('RZR302_Day6', 'Afternoon'), ('RZR302_Day6', 'Evening'), ('RZR302_Day2', 'Morning'), ('RZR302_Day2', 'Evening'), ('RZR302_Day7', 'Morning'), ('RZR302_Day7', 'Afternoon'), ('RZR302_Day7', 'Evening'), ('RZR302_Day3', 'Morning'), ('RZR302_Day3', 'Evening'), ('RZR304_Day5', 'Morning'), ('RZR304_Day5', 'Afternoon'), ('RZR304_Day5', 'Evening'), ('RZR304_Day1', 'Morning'), ('RZR205_Day3', 'Morning'), ('RZR205_Day3', 'Afternoon'), ('RZR205_Day3', 'Evening'), ('RZR227_Day5', 'Morning'), ('RZR227_Day5', 'Afternoon'), ('RZR227_Day5', 'Evening'), ('RZR227_Day1', 'Morning'), ('RZR227_Day1', 'Afternoon'), ('RZR227_Day1', 'Evening'), ('R

In [18]:
print(online_slots)

[('Online_Day1', 'Morning'), ('Online_Day1', 'Afternoon'), ('Online_Day1', 'Evening'), ('Online_Day2', 'Morning'), ('Online_Day2', 'Afternoon'), ('Online_Day2', 'Evening'), ('Online_Day3', 'Morning'), ('Online_Day3', 'Afternoon'), ('Online_Day3', 'Evening'), ('Online_Day4', 'Morning'), ('Online_Day4', 'Afternoon'), ('Online_Day4', 'Evening'), ('Online_Day5', 'Morning'), ('Online_Day5', 'Afternoon'), ('Online_Day5', 'Evening'), ('Online_Day6', 'Morning'), ('Online_Day6', 'Afternoon'), ('Online_Day6', 'Evening'), ('Online_Day7', 'Morning'), ('Online_Day7', 'Afternoon'), ('Online_Day7', 'Evening')]


In [12]:
# Assuming room_capacity, inperson_slots, and online_slots are already defined as per your setup

# Initialize an empty dictionary to store slot capacities
slot_capacities = {}

# Merge in-in_person_slots slots with capacities
for slot in in_person_slots:
    # Extract the room identifier from the slot (before "_Day")
    room = slot[0].split('_Day')[0]
    # Use the room to get the capacity from room_capacity
    capacity = room_capacity.get(room, 0)  # Default to 0 if room not found
    # Add the slot and its capacity to the dictionary
    slot_capacities[slot] = capacity

# Merge online slots with capacities
# Assuming infinite capacity for online slots
for slot in online_slots:
    slot_capacities[slot] = float('inf')

# Now slot_capacities contains all slots as keys and their capacities as values
print(slot_capacities)

{('RZR227_Day7', 'Morning'): 12, ('RZR227_Day7', 'Afternoon'): 12, ('RZR227_Day7', 'Evening'): 12, ('RZR227_Day3', 'Morning'): 12, ('RZR227_Day3', 'Afternoon'): 12, ('RZR227_Day3', 'Evening'): 12, ('RZR302_Day5', 'Morning'): 12, ('RZR302_Day5', 'Evening'): 12, ('RZR302_Day1', 'Morning'): 12, ('RZR302_Day1', 'Evening'): 12, ('RZR302_Day4', 'Morning'): 12, ('RZR302_Day4', 'Evening'): 12, ('RZR302_Day6', 'Morning'): 12, ('RZR302_Day6', 'Afternoon'): 12, ('RZR302_Day6', 'Evening'): 12, ('RZR302_Day2', 'Morning'): 12, ('RZR302_Day2', 'Evening'): 12, ('RZR302_Day7', 'Morning'): 12, ('RZR302_Day7', 'Afternoon'): 12, ('RZR302_Day7', 'Evening'): 12, ('RZR302_Day3', 'Morning'): 12, ('RZR302_Day3', 'Evening'): 12, ('RZR304_Day5', 'Morning'): 20, ('RZR304_Day5', 'Afternoon'): 20, ('RZR304_Day5', 'Evening'): 20, ('RZR304_Day1', 'Morning'): 20, ('RZR205_Day3', 'Morning'): 20, ('RZR205_Day3', 'Afternoon'): 20, ('RZR205_Day3', 'Evening'): 20, ('RZR227_Day5', 'Morning'): 12, ('RZR227_Day5', 'Afternoon'

In [13]:
#convert to values in write up
S = list(schedule_counts.keys()) #set of all unique schedules
N_t = schedule_counts      #num of people with a given unique schedule
E = set(set_of_exams)
E_p = set(in_person_exams)
E_o = online_exams
d_E = exam_demand
R = all_slots
R_p = in_person_slots
R_o = online_slots
c_R = slot_capacities

In [14]:
large = {'PSYC101', 'CHEM121'}
E = E - large
E_p = E_p - large

In [15]:
# Define the first stage model

day_pairs = range(1, 7)

x_vars = [(slot,exam) for slot in R for exam in E]

y_vars = [(schedule, day) for schedule in S for day in day_pairs]

z_vars = [(slot, exam) for slot in R_p for exam in E_p]

model1 = gp.Model("First Stage Model")

# Add variables
x = model1.addVars(x_vars, vtype=GRB.BINARY, name="x variables")

y = model1.addVars(y_vars, lb = 0, name="y variables") #Relaxed into continuous vars

z = model1.addVars(z_vars, lb = 0, name="z variables") #Relaxed into continuous vars

# Add Objective Function

#obj1 = gp.quicksum(N_t[s] * gp.quicksum(y_vars[s,d] for d in day_pairs) for s in S)
obj1 = gp.quicksum(N_t[s] * gp.quicksum(y[s, d] for d in day_pairs) for s in S)


obj2 = gp.quicksum(z[r,e] for r in R_p for e in E_p)

model1.setObjective(obj1 + obj2, GRB.MINIMIZE)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-01-24


In [16]:
# Add Constraints

#capacity constraints & capacity relaxation measure
for exam in E_p: 
    for slot in R_p:
        if c_R[slot] <= d_E[exam]: 
            model1.addConstr(x[slot,exam] == 0, name=f"capacity_constraint_{slot}_{exam}" ) 
            #model1.addConstr(z[slot,exam] == 0, name = "capacity_relaxation")
        else: 
            model1.addConstr(z[slot,exam] >= (2*d_E[exam] - c_R[slot])*x[slot,exam],
             name = f"capacity_relaxation_{slot}_{exam}")


for exam in E_p: 
    for slot in R_o:
        model1.addConstr(x[slot,exam] == 0, name = f"in_person_constraint_{slot}_{exam}")

for exam in E_o:
    for slot in R_p:
        model1.addConstr(x[slot,exam] == 0, name = f"online_constraint_{slot}_{exam}")
    
for exam in E:
    model1.addConstr(gp.quicksum(x[slot,exam] for slot in R) == 1, name = f"one_slot_per_exam_{exam}")

for r in R_p:
    model1.addConstr(gp.quicksum(x[r,e] for e in E_p) <= 1, name = f"at_most_one_in_person_{r}")

#
""" for i in range(1, 7):
    consecutive_slots = []
    for j in R:
        slot_day = j[0].split('_')[1]
        if (slot_day == f'Day{i}') or (slot_day == f'Day{i+1}'):
            consecutive_slots.append(j)
    for schedule in S:
        model1.addConstr(gp.quicksum(x[(r, e)] for r in consecutive_slots for e in set(schedule).intersection(E)) <= 2 + y[schedule, i], "y Violations") """

for i in range(1, 7):
    consecutive_slots = []
    for j in R:
        slot_day = j[0].split('_')[1]
        if (slot_day == f'Day{i}') or (slot_day == f'Day{i+1}'):
            consecutive_slots.append(j)
    for schedule in S:
        required_exams = set(schedule).intersection(E)
        if required_exams:
            # Ensure that the sum of x for required exams in consecutive slots is not more than 2 plus the corresponding y value
            model1.addConstr(
                gp.quicksum(x[r, e] for r in consecutive_slots for e in required_exams) <= 2 + y[schedule, i],
                name=f"y_Violations_schedule{S.index(schedule)}_day{i}"
            )

In [18]:
model1.setParam('TimeLimit', 300)
model1.optimize()
print(model1.status)

if model1.status == GRB.OPTIMAL:
    print('Optimal solution found.')

    # Retrieve the values of the variables in the optimal solution
    x_values = model1.getAttr('X', x)
    y_values = model1.getAttr('X', y)
    z_values = model1.getAttr('X', z)

    # Print the values of the variables
    print('x variables:')
    x_dict = {}
    for var in x_values:
        x_dict[var] =x_values[var]

    print('y variables:')
    y_dict = {}
    for var in y_values:
        y[var] = y_values[var]
    print('z variables:')
    z_dict = {}
    for var in z_values:
        z_dict[var] = z_values[var]


if model1.status == GRB.INFEASIBLE:
    model1.computeIIS()

    # Print out the constraints that are part of the IIS
    for c in model1.getConstrs():
        if c.IISConstr:
            print('Infeasible constraint:', c.constrName)

Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[rosetta2])

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 528867 rows, 979622 columns and 33415627 nonzeros
Model fingerprint: 0xadbd48ae
Variable types: 489622 continuous, 490000 integer (490000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [1e+00, 6e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]
Presolved: 9701 rows, 331080 columns, 5078731 nonzeros

Continuing optimization...


Cutting planes:
  MIR: 4
  Zero half: 10

Explored 1 nodes (61590 simplex iterations) in 3.42 seconds (0.00 work units)
Thread count was 8 (of 8 available processors)

Solution count 9: 1132 1137 1140 ... 4055

Optimal solution found (tolerance 1.00e-04)
Best objective 1.132000000000e+03, best bound 1.132000000000e+03, gap 0.0000%
2
Optimal solution found.
x variables:
y variables:
z variables:


In [ ]:
x_values = model1.getAttr('X', x)
# y_values = model1.getAttr('X', y)
z_values = model1.getAttr('X', z)

# Print the values of the variables
print('x variables:')
x_dict = {}
for var in x_values:
    x_dict[var] =x_values[var]

print('y variables:')
y_dict = {}
for var in y_values:
    y[var] = y_values[var]

print('z variables:')
z_dict = {}
for var in z_values:
    z_dict[var] = z_values[var]

keys_equal_to_1 = set([key[1] for key, value in x_dict.items() if value == 1])
keys_equal_to_1

x variables:
y variables:
z variables:


{'ANTH200',
 'ANTH205',
 'ARCH311',
 'ARCH423',
 'ASIA322',
 'ASTR101',
 'ASTR102',
 'ASTR350',
 'ASTR451',
 'ASTR470',
 'BIOE123',
 'BIOE252',
 'BIOE341',
 'BIOE383',
 'BIOE385',
 'BIOE420',
 'BIOE422',
 'BIOE440',
 'BIOE446',
 'BIOE449',
 'BIOE451',
 'BIOS122',
 'BIOS201',
 'BIOS202',
 'BIOS300',
 'BIOS301',
 'BIOS312',
 'BIOS329',
 'BIOS341',
 'BIOS344',
 'BIOS352',
 'BIOS353',
 'BIOS372',
 'BIOS385',
 'BIOS441',
 'BUSI222',
 'BUSI224',
 'BUSI305',
 'BUSI310',
 'BUSI343',
 'BUSI390',
 'BUSI395',
 'BUSI401',
 'BUSI430',
 'BUSI440',
 'BUSI461',
 'CEVE211',
 'CEVE302',
 'CEVE325',
 'CEVE400',
 'CEVE426',
 'CEVE431',
 'CEVE460',
 'CEVE471',
 'CEVE476',
 'CHBE301',
 'CHBE390',
 'CHBE401',
 'CHBE412',
 'CHBE418',
 'CHBE420',
 'CHBE468',
 'CHBE470',
 'CHEM201',
 'CHEM211',
 'CHEM219',
 'CHEM313',
 'CHEM401',
 'CHEM430',
 'CHEM452',
 'CHEM462',
 'CHEM496',
 'CMOR302',
 'CMOR304',
 'CMOR360',
 'CMOR405',
 'CMOR410',
 'CMOR420',
 'CMOR422',
 'CMOR441',
 'CMOR451',
 'CMOR462',
 'CMOR494',
 'CM

In [ ]:
keys_equal_to_1 = [key for key, value in z_dict.items() if value != 0]
keys_equal_to_1

[(('HRZAMP_Day5', 'Morning'), 'MATH212'),
 (('HRZAMP_Day5', 'Afternoon'), 'MATH102'),
 (('HRZAMP_Day5', 'Evening'), 'MATH101'),
 (('HRZAMP_Day1', 'Afternoon'), 'PSYC203'),
 (('HRZAMP_Day1', 'Evening'), 'BUSI310'),
 (('HRZAMP_Day4', 'Afternoon'), 'PHYS125'),
 (('HRZAMP_Day4', 'Evening'), 'BIOS341'),
 (('HRZAMP_Day7', 'Morning'), 'STAT310'),
 (('HRZAMP_Day7', 'Afternoon'), 'CHEM211'),
 (('HRZAMP_Day7', 'Evening'), 'MATH355'),
 (('HRZAMP_Day3', 'Afternoon'), 'PHYS101')]

In [ ]:
# Assuming initial setups are as previously defined
""" E = set(...)  # Set of all offered exams
E_p = set(...)  # Set of in-person offered exams
d_E = {...}  # Dictionary mapping each exam to its demand
S = [...]  # List of unique schedules (each schedule is a set of exam codes)
N_t = {...}  # Dictionary mapping each unique schedule to the number of students """
#convert to values in write up
S = list(schedule_counts.keys()) #set of all unique schedules
N_t = schedule_counts      #num of people with a given unique schedule
E = set(set_of_exams)
E_p = set(in_person_exams)
E_o = online_exams
d_E = exam_demand
R = all_slots
R_p = in_person_slots
R_o = online_slots
c_R = slot_capacities
divide_exams = ['PSYC101', 'CHEM121', 'MATH212', 'STAT310', 'MATH102', 'CHEM211', 'MATH101', 'PHYS125', 'BIOS341', 
                'BUSI310', 'PSYC203', 'MATH355', 'PHYS101']  # Exams to be divided into Part A and B


exam_counters = {exam: 0 for exam in divide_exams}  # Tracks encounters for dividing

for E_X in divide_exams:
    # Update E and E_p for each divided exam
    E.discard(E_X)
    E.update({f'{E_X}_A', f'{E_X}_B'})
    if E_X in E_p:
        E_p.discard(E_X)
        E_p.update({f'{E_X}_A', f'{E_X}_B'})

    # Update d_E
    if E_X in d_E:
        VAL = d_E[E_X] / 2
        d_E[f'{E_X}_A'] = VAL
        d_E[f'{E_X}_B'] = VAL
        del d_E[E_X]

# Update S and N_t with alternating Part A and Part B assignments
new_S = []
new_N_t = {}

for schedule in S:
    updated_schedule = set(schedule)  # Start with a copy of the current schedule
    for E_X in divide_exams:
        if E_X in updated_schedule:
            # Determine whether to use Part A or Part B based on the current count
            part_suffix = '_A' if exam_counters[E_X] % 2 == 0 else '_B'
            updated_schedule.discard(E_X)
            updated_schedule.add(f'{E_X}{part_suffix}')

            # Update the exam counter for this divided exam
            exam_counters[E_X] += 1

    # Convert updated_schedule set to a tuple for use as a dictionary key
    schedule_tuple = tuple(sorted(updated_schedule))  # Sorting to ensure consistency
    new_S.append(updated_schedule)
    # Update or add the new schedule in new_N_t, accumulating counts if necessary
    if schedule_tuple in new_N_t:
        new_N_t[schedule_tuple] += N_t[tuple(sorted(schedule))]
    else:
        new_N_t[schedule_tuple] = N_t[tuple(sorted(schedule))]

# Replace old S and N_t with new ones, converting new_S sets to sorted tuples
S = [tuple(sorted(sch)) for sch in new_S]
N_t = new_N_t

In [ ]:
day_pairs = range(1, 7)

x_vars = [(slot,exam) for slot in R for exam in E]

y_vars = [(schedule, day) for schedule in S for day in day_pairs]

z_vars = [(slot, exam) for slot in R_p for exam in E_p]

model1 = gp.Model("First Stage Model")

# Add variables
x = model1.addVars(x_vars, vtype=GRB.BINARY, name="x variables")

y = model1.addVars(y_vars, lb = 0, name="y variables") #Relaxed into continuous vars

z = model1.addVars(z_vars, lb = 0, name="z variables") #Relaxed into continuous vars

# Add Objective Function

#obj1 = gp.quicksum(N_t[s] * gp.quicksum(y_vars[s,d] for d in day_pairs) for s in S)
obj1 = gp.quicksum(N_t[s] * gp.quicksum(y[s, d] for d in day_pairs) for s in S)


obj2 = gp.quicksum(z[r,e] for r in R_p for e in E_p)

model1.setObjective(obj1 + obj2, GRB.MINIMIZE)

# Add Constraints

#capacity constraints & capacity relaxation measure
for exam in E_p: 
    for slot in R_p:
        if c_R[slot] <= d_E[exam]: 
            model1.addConstr(x[slot,exam] == 0, name=f"capacity_constraint_{slot}_{exam}" ) 
            #model1.addConstr(z[slot,exam] == 0, name = "capacity_relaxation")
        else: 
            model1.addConstr(z[slot,exam] >= (2*d_E[exam] - c_R[slot])*x[slot,exam],
             name = f"capacity_relaxation_{slot}_{exam}")


for exam in E_p: 
    for slot in R_o:
        model1.addConstr(x[slot,exam] == 0, name = f"in_person_constraint_{slot}_{exam}")

for exam in E_o:
    for slot in R_p:
        model1.addConstr(x[slot,exam] == 0, name = f"online_constraint_{slot}_{exam}")
    
for exam in E:
    model1.addConstr(gp.quicksum(x[slot,exam] for slot in R) == 1, name = f"one_slot_per_exam_{exam}")

for r in R_p:
    model1.addConstr(gp.quicksum(x[r,e] for e in E_p) <= 1, name = f"at_most_one_in_person_{r}")


for i in range(1, 7):
    consecutive_slots = []
    for j in R:
        slot_day = j[0].split('_')[1]
        if (slot_day == f'Day{i}') or (slot_day == f'Day{i+1}'):
            consecutive_slots.append(j)
    for schedule in S:
        required_exams = set(schedule).intersection(E)
        if required_exams:
            # Ensure that the sum of x for required exams in consecutive slots is not more than 2 plus the corresponding y value
            model1.addConstr(
                gp.quicksum(x[r, e] for r in consecutive_slots for e in required_exams) <= 2 + y[schedule, i],
                name=f"y_Violations_schedule{S.index(schedule)}_day{i}"
            )

In [ ]:
model1.setParam('TimeLimit', 300)
model1.optimize()

Set parameter TimeLimit to value 300
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[rosetta2])

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 558504 rows, 1038107 columns and 35834812 nonzeros
Model fingerprint: 0x3647057e
Variable types: 518707 continuous, 519400 integer (519400 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [1e+00, 6e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]
Found heuristic solution: objective 4778.0000000
Presolve removed 328102 rows and 252216 columns (presolve time = 6s) ...
Presolve removed 530782 rows and 443262 columns (presolve time = 11s) ...
Presolve removed 530782 rows and 684403 columns (presolve time = 15s) ...
Presolve removed 530782 rows and 684403 columns (presolve time = 20s) ...
Presolve removed 539290 rows and 692935 columns (presolve time = 30s) ...
Presolve removed 539290 rows and 692935 columns (pr

In [ ]:
if model1.status == GRB.OPTIMAL or model1.SolCount > 0:
    print('Optimal or best solution found.')

    # Retrieve the values of the variables in the optimal solution
    x_values = model1.getAttr('X', x)
    y_values = model1.getAttr('X', y)
    z_values = model1.getAttr('X', z)

    # Print the values of the variables
    #print('x variables:')
    x_dict = {}
    for var in x_values:
        x_dict[var] =x_values[var]

    #print('y variables:')
    y_dict = {}
    for var in y_values:
        y[var] = y_values[var]

    #print('z variables:')
    z_dict = {}
    for var in z_values:
        z_dict[var] = z_values[var]


if model1.status == GRB.INFEASIBLE:
    model1.computeIIS()

    # Print out the constraints that are part of the IIS
    for c in model1.getConstrs():
        if c.IISConstr:
            print('Infeasible constraint:', c.constrName)

Optimal or best solution found.


AttributeError: 'float' object has no attribute '_typeenum'

In [55]:
x_values = model1.getAttr('X', x)
# y_values = model1.getAttr('X', y)
z_values = model1.getAttr('X', z)

# Print the values of the variables
print('x variables:')
x_dict = {}
for var in x_values:
    x_dict[var] =x_values[var]

print('y variables:')
y_dict = {}
for var in y_values:
    y[var] = y_values[var]

print('z variables:')
z_dict = {}
for var in z_values:
    z_dict[var] = z_values[var]

keys_equal_to_1 = set([key[1] for key, value in z_dict.items() if value == 1])
keys_equal_to_1

x variables:
y variables:
z variables:


set()

In [ ]:
#Q = {{comp140_1, comp140_2}, {psyc_1, psyc_part2} ,,, }

In [53]:
from collections import defaultdict

exam_date_map = {}
for slot, cls in in_person_slots:
    # Extract day number and convert it to an integer
    day_num = int(slot.split('Day')[-1])
    exam_date_map[cls] = day_num

student_exam_dates = defaultdict(set)
for idx, classes in enumerate(S):
    for cls in classes:  # Iterating through each class in the tuple
        if cls in exam_date_map:
            student_exam_dates[idx].add(exam_date_map[cls])

# Re-identify students with exams on consecutive days, using the adjusted mapping
students_with_consecutive_exams = 0
for exam_dates in student_exam_dates.values():
    sorted_dates = sorted(exam_dates)
    for i in range(len(sorted_dates) - 1):
        if sorted_dates[i] + 1 == sorted_dates[i + 1]:
            students_with_consecutive_exams += 1
            break

print("Number of Students with at least 2 exams on back to back days:", students_with_consecutive_exams)
        

#number of studnets with an exam on the same day

students_with_same_day_exams = 0

# Iterate through each student's schedule
for classes in S:
    # Aggregate exam days for this student
    student_exam_days = defaultdict(int)  # Default to 0 for any new day
    for cls in classes:
        if cls in exam_date_map:
            student_exam_days[exam_date_map[cls]] += 1  # Increment the count for the exam day of this class
    
    # Check if the student has more than one exam on any day
    if any(count > 1 for count in student_exam_days.values()):
        students_with_same_day_exams += 1

print("Number of Students with an exam on the same day:", students_with_same_day_exams)

#number of students with overlaps

# First, we need to map exams not just to days but to day and time slots to check for time conflicts
exam_datetime_map = {}
for slot, cls in in_person_slots:
    # Convert day and time to a unique identifier for simplicity
    day_time = f"{slot[0]}_{slot[1]}"  # Combines day and time into a single string
    exam_datetime_map[cls] = day_time

# Map students to the day and time slots of their exams
student_exam_datetimes = defaultdict(set)
for idx, classes in enumerate(S):
    for cls in classes:
        if cls in exam_datetime_map:
            student_exam_datetimes[idx].add(exam_datetime_map[cls])

# Identify students with time conflicts in their exams
students_with_time_conflicts = 0
for exam_datetimes in student_exam_datetimes.values():
    if len(exam_datetimes) != len(set(exam_datetimes)):
        students_with_time_conflicts += 1

print("Number of Students with an exam at the same time of another exam:", students_with_time_conflicts)

total_days_between_exams = 0
total_intervals_counted = 0

for exam_dates in student_exam_dates.values():
    sorted_dates = sorted(exam_dates)
    for i in range(len(sorted_dates) - 1):
        days_between = sorted_dates[i + 1] - sorted_dates[i]
        total_days_between_exams += days_between
        total_intervals_counted += 1

# Calculate the average, avoiding division by zero
average_days_between_exams = (total_days_between_exams / total_intervals_counted) if total_intervals_counted else 0

print("Average number of days between two exams", average_days_between_exams)


Number of Students with at least 2 exams on back to back days: 0
Number of Students with an exam on the same day: 0
Number of Students with an exam at the same time of another exam: 0
Average number of days between two exams 0
